In [ ]:
%pip install -q pyyaml tiktoken

In [ ]:
# Manual format validation
import json
DATASET_PATH = 'datasets/merged_assembly_dataset.jsonl'

def show_properties(filename):
    print(f"{filename}:")
    with open(filename, 'r') as file:
        line = file.readline()
        properties = json.loads(line)
        for key, value in properties.items():
            print(f"- {key}: {str(value)[:50].replace('\n', ' ')}")
            
show_properties(DATASET_PATH)

In [ ]:
# Search invalid items

from collections import Counter
modifiers = Counter()
categories = Counter()
models = Counter()

with open(DATASET_PATH, 'r') as file:
    for line in file:
        properties = json.loads(line)
        modifiers[properties['modifier']] += 1
        categories[properties['category']] += 1
        assert properties['non_optimized_simplified_source_linked_assembly'].lower() != 'none'
        assert properties['simplified_source_linked_assembly'].lower() != 'none'
        assert properties['source_linked_assembly'].lower() != 'none'
        models[properties['model']] += 1

modifiers, categories, models

Final fields:

```yaml
id
prompt_id
category
modifier
preprompt
prompt
model
go-code-data
go-code-length
go-code-lines
go-code-tokens
asm-base-data
asm-base-length
asm-base-lines
asm-base-tokens
asm-clean-data
asm-clean-length
asm-clean-lines
asm-clean-tokens
asm-noopt-data
asm-noopt-length
asm-noopt-lines
asm-noopt-tokens
```

In [ ]:
# Generate output file
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

cid = 0
with open(DATASET_PATH, 'r') as inf:
    with open('datasets/llm-goasm-dataset.jsonl', 'w') as outf:
        for line in inf:
            cid += 1
            properties = json.loads(line)
            data = {
                "id": cid,
                "prompt_id": properties['prompt_id'],
                "category": properties['category'],
                "modifier": properties['modifier'],
                "preprompt": properties['preprompt'],
                "prompt": properties['prompt'],
                "model": properties['model'],
                
                "go-code-data": properties['code'],
                "go-code-length": len(properties['code']),
                "go-code-lines": properties['code'].count('\n') + 1,
                "go-code-tokens": len(tokenizer.encode(properties['code'])),
                
                "asm-base-data": properties['source_linked_assembly'],
                "asm-base-length": len(properties['source_linked_assembly']),
                "asm-base-lines": properties['source_linked_assembly'].count('\n') + 1,
                "asm-base-tokens": len(tokenizer.encode(properties['source_linked_assembly'])),
                
                "asm-clean-data": properties['simplified_source_linked_assembly'],
                "asm-clean-length": len(properties['simplified_source_linked_assembly']),
                "asm-clean-lines": properties['simplified_source_linked_assembly'].count('\n') + 1,
                "asm-clean-tokens": len(tokenizer.encode(properties['simplified_source_linked_assembly'])),
                
                "asm-noopt-data": properties['non_optimized_simplified_source_linked_assembly'],
                "asm-noopt-length": len(properties['non_optimized_simplified_source_linked_assembly']),
                "asm-noopt-lines": properties['non_optimized_simplified_source_linked_assembly'].count('\n') + 1,
                "asm-noopt-tokens": len(tokenizer.encode(properties['non_optimized_simplified_source_linked_assembly'])),
            }
            outf.write(json.dumps(data) + '\n')